# Liu2024 — TWFB + DGFMDM (FgMDM), faithful + honest, with a reusable covariance cache

A runnable port of the Liu2024 strongest baseline. Same features and classifier as the shipped MATLAB
(`fgmdm` = FgMDM, **no LTSA**), with four protocols and a structured, shareable per-trial covariance cache.

**Four protocols** (toggle in CONFIG):
- `shipped_fb_leaky`  — 8 bands, full 0–4 s window, `max`-over-bands on the **test** set (reproduces `TWFB_DGFMDM.m`).
- `shipped_fb_honest` — same 8 bands, but the band is chosen by **inner CV on train only** (leakage-free).
- `twfb_leaky`        — 8 bands × 7 windows, `max`-over-combos on the **test** set (the ~72% oracle number).
- `twfb_honest`       — same 56 combos, view chosen by **inner CV on train only** (the honest ceiling).
- `perband_fixed`     — each band, no selection, honest 60/40 (a no-cherry-picking floor).

**Honest = selection after training, never on test.** For each outer 60/40 split, an inner 3-fold CV runs
*inside the training trials*; the view with the best mean validation balanced-accuracy is chosen, then scored
once on the untouched test fold. Leaky instead chooses the view by its test accuracy.

**Covariance cache (reusable).** Per-trial covariances are the expensive part; this notebook caches them so all
protocols (and other notebooks / collaborators) can reuse them instantly. See `COVARIANCE_CACHE_README.md`.
- Layout: `{covariance_cache_dir}/cov_{signature}/sub-XX_cov.npz` + `cache_manifest.json`.
- Keys inside each npz: `fb__{lo}_{hi}` (full 0–4 s cov, shape `(n_trials,29,29)`), `tw__{lo}_{hi}__{wstart}`
  (1-s window cov), plus `y`, `onsets`, `_cov_sig`.
- `covariance_cache_mode`: `auto` | `readonly` | `rebuild` | `off`. The `signature` hashes every
  covariance-affecting setting (channels, filtering, bands, windows, regularization), so incompatible caches
  never collide.

> Logging/artifacts mirror `liu2024_source_mat_sjepa_prelocal_augmented`: timestamped `print`→`run.log`,
> `RUN_ID` from CONFIG hash, `config.json`, and the CSV/JSON artifact set. **Edit the single CONFIG cell (or
> apply a sweep JSON) and re-run.**

# 1. Setup

In [ ]:
import os, sys, glob, json, time, random, hashlib, builtins, platform, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import butter, iirnotch, lfilter, filtfilt

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as e:
    HAVE_MPL = False; print("matplotlib unavailable:", e)

from pyriemann.classification import FgMDM     # == MATLAB fgmdm (DGFMDM)

warnings.filterwarnings("ignore")
print("python", platform.python_version(), "| numpy", np.__version__)
import pyriemann; print("pyriemann", pyriemann.__version__)

# 2. CONFIG  *(edit this one cell, or apply a sweep JSON, then re-run)*

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ---- identity ----
    "experiment_name": "twfb_dgfmdm_faithful_v2",
    "config_note":     "4 protocols (fb/twfb x leaky/honest) + covariance cache",

    # ---- paths ----
    "source_roots": [
        str(WORKING_DIR.parent / "Liu2024_matlab_code" / "sourcedata"),
        str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    ],
    "artifact_dir":         str(WORKING_DIR / "artifacts" / "liu2024-twfb-dgfmdm-faithful-v2"),
    "covariance_cache_dir": str(WORKING_DIR / "artifacts" / "covariance_cache"),
    "covariance_cache_mode": "auto",     # 'auto' | 'readonly' | 'rebuild' | 'off'

    # ---- dataset / channels (faithful to TWFB_DGFMDM.m) ----
    "native_sfreq": 500,
    "channel_indices": list(range(0,17)) + list(range(18,30)),  # [1:17 19:30] -> 29 ch, drop CPz
    "marker_channel_index": 32,
    "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300],
    "onset_fallback_sample": 1003,
    "subjects_to_use": None,             # None=all; or e.g. list(range(1,11))

    # ---- windowing (faithful: 0-4 s after onset, 800-sample warm-up) ----
    "preroll_samples": 800,
    "mi_window_samples": 2000,

    # ---- filtering ----
    "notch_freq": 50.0, "notch_Q": 6.0,
    "butter_order": 2,
    "filter_phase": "causal",            # 'causal' (one-pass, matches MATLAB 'filter') | 'zero' (filtfilt)
    "freq_bands": [[8,12],[8,20],[8,30],[12,20],[15,20],[15,30],[20,30],[8,15]],  # the 8 shipped bands

    # ---- time-window search (added on top; absent from the shipped .m) ----
    "time_window_starts_s": [0.0,0.5,1.0,1.5,2.0,2.5,3.0],
    "time_window_len_s": 1.0,

    # ---- covariance ----
    "cov_trace_normalize": True,         # set False + cov_shrinkage 0 for bit-for-bit raw scatter
    "cov_shrinkage": 0.10,
    "fgmdm_metric": "riemann",           # 'riemann' | 'logeuclid'

    # ---- protocols to run ----
    "run_shipped_fb_leaky":  True,
    "run_shipped_fb_honest": True,
    "run_twfb_leaky":        True,
    "run_twfb_honest":       True,
    "run_perband_fixed":     True,

    # ---- evaluation ----
    "cv_scheme": "repeated_holdout",     # 'repeated_holdout' (60/40 x n_repeats) | 'kfold5'
    "n_repeats": 10,
    "train_size": 24, "test_size": 16,
    "honest_test_frac": 0.40,
    "n_splits": 5,                       # used when cv_scheme='kfold5'
    "inner_folds": 3,
    "random_state": 2026,
}

# ---- derived ----
BANDS = [tuple(b) for b in CONFIG["freq_bands"]]
FS    = CONFIG["native_sfreq"]
CH    = CONFIG["channel_indices"]
NCH   = len(CH)
WIN_STARTS = CONFIG["time_window_starts_s"]
print(f"channels={NCH}  bands={len(BANDS)}  windows={len(WIN_STARTS)}  TWxFB combos={len(BANDS)*len(WIN_STARTS)}")
print(f"protocols: fb_leaky={CONFIG['run_shipped_fb_leaky']} fb_honest={CONFIG['run_shipped_fb_honest']} "
      f"twfb_leaky={CONFIG['run_twfb_leaky']} twfb_honest={CONFIG['run_twfb_honest']} | cache={CONFIG['covariance_cache_mode']}")

## 2.1 Logging, Run ID, Covariance-cache signature

In [ ]:
def _cov_signature(cfg):
    keys = ["native_sfreq","channel_indices","marker_channel_index","onset_marker_value",
            "onset_plausible_range","onset_fallback_sample","preroll_samples","mi_window_samples",
            "notch_freq","notch_Q","butter_order","filter_phase","freq_bands",
            "time_window_starts_s","time_window_len_s","cov_trace_normalize","cov_shrinkage"]
    subset = {k: cfg[k] for k in keys}
    h = hashlib.md5(json.dumps(subset, sort_keys=True, default=str).encode()).hexdigest()[:10]
    return h, subset

def create_run_id():
    h = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{datetime.now().strftime('%Y%m%d_%H%M')}_{h}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
COV_SIG, COV_SUBSET = _cov_signature(CONFIG)
COV_DIR = Path(CONFIG["covariance_cache_dir"]) / f"cov_{COV_SIG}"

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _w(stream, t):
    try: stream.write(t)
    except UnicodeEncodeError:
        enc=getattr(stream,"encoding",None) or "utf-8"; stream.write(t.encode(enc,"replace").decode(enc,"replace"))
_ORIG_PRINT = builtins.print
def _tprint(*a, **k):
    sep=k.pop("sep"," "); end=k.pop("end","\n"); k.pop("flush",False); k.pop("file",None)
    msg=sep.join(str(x) for x in a); lead=len(msg)-len(msg.lstrip("\n")); body=msg[lead:]
    if lead: _w(sys.stdout,"\n"*lead); _w(_LOG,"\n"*lead)
    if body:
        s=f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {body}"; _w(sys.stdout,s+end); _w(_LOG,s+end)
    else: _w(sys.stdout,end); _w(_LOG,end)
builtins.print=_tprint
random.seed(CONFIG["random_state"]); np.random.seed(CONFIG["random_state"])
with open(ARTIFACT_DIR/"config.json","w") as f: json.dump(CONFIG,f,indent=2,default=str)

print("="*70)
print(f"Experiment: {CONFIG['experiment_name']}")
print(f"Note:       {CONFIG['config_note']}")
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Cov cache:  {COV_DIR}  (mode={CONFIG['covariance_cache_mode']}, sig={COV_SIG})")
print("="*70)

# 3. Data loading + MI onset

In [ ]:
def find_subject_files(roots):
    for root in roots:
        files = sorted(glob.glob(os.path.join(root, "sub-*", "sub-*_eeg.mat")))
        if files:
            print(f"using source root: {root}  ({len(files)} subjects)")
            return files
    raise FileNotFoundError(f"No sub-*_eeg.mat under any of: {roots}")

def load_subject_raw(path):
    """Return rawdata (n,33,4000), labels (n,) in {0,1}, per-trial onset sample list."""
    m = sio.loadmat(path); eeg = m["eeg"][0,0]
    raw = np.asarray(eeg["rawdata"], dtype=np.float64)
    lab = np.asarray(eeg["label"]).ravel().astype(int)
    if set(np.unique(lab)).issubset({1,2}): lab = lab - 1
    mark = raw[:, CONFIG["marker_channel_index"], :]
    lo, hi = CONFIG["onset_plausible_range"]
    first = []
    for t in range(raw.shape[0]):
        idx = np.where(mark[t] == CONFIG["onset_marker_value"])[0]
        in_range = idx[(idx >= lo) & (idx <= hi)]
        first.append(int(in_range[0]) if in_range.size else (int(idx[0]) if idx.size else -1))
    plausible = [o for o in first if lo <= o <= hi]
    med = int(np.median(plausible)) if plausible else CONFIG["onset_fallback_sample"]
    onsets = [o if lo <= o <= hi else med for o in first]
    return raw, lab, np.asarray(onsets, dtype=int)
print("Loader defined.")

# 4. Covariance construction + reusable cache

In [ ]:
def _apply(b, a, x):
    return filtfilt(b, a, x, axis=1) if CONFIG["filter_phase"] == "zero" else lfilter(b, a, x, axis=1)

def _regularize(c):
    if CONFIG["cov_trace_normalize"]:
        c = c / np.trace(c)
    g = CONFIG["cov_shrinkage"]
    if g > 0:
        scale = (1.0/NCH) if CONFIG["cov_trace_normalize"] else (np.trace(c)/NCH)
        c = (1-g)*c + g*scale*np.eye(NCH)
    return c

def _band_segment(raw, onsets, bd):
    """Faithful 0-4 s MI segment for one band: (n,29,mi_window_samples). Notch + bandpass on a warm-up pad."""
    wo = CONFIG["notch_freq"]/(FS/2); bw = wo/CONFIG["notch_Q"]
    nb, na = iirnotch(wo, wo/bw)
    bb, ba = butter(CONFIG["butter_order"], [bd[0]/(FS/2), bd[1]/(FS/2)], btype="band")
    pre = CONFIG["preroll_samples"]; L = CONFIG["mi_window_samples"]; n = raw.shape[0]
    out = np.zeros((n, NCH, L))
    for t in range(n):
        s0 = onsets[t] - pre
        seg = raw[t][:, s0:s0+pre+L][CH, :]
        seg = _apply(nb, na, seg)
        seg = _apply(bb, ba, seg)
        out[t] = seg[:, pre:pre+L]
    return out

def _cov(seg_slice):
    n = seg_slice.shape[0]; C = np.zeros((n, NCH, NCH))
    for t in range(n):
        X = seg_slice[t]; C[t] = _regularize(X @ X.T)
    return C

def build_subject_covariances(raw, onsets):
    """All views for one subject. Keys: 'fb__lo_hi' (full 0-4s) and 'tw__lo_hi__wstart' (1-s window)."""
    views = {}
    wlen = int(round(CONFIG["time_window_len_s"]*FS))
    for bd in BANDS:
        seg = _band_segment(raw, onsets, bd)                       # (n,29,L)
        views[f"fb__{bd[0]}_{bd[1]}"] = _cov(seg)                   # full window
        for ws in WIN_STARTS:
            s = int(round(ws*FS))
            views[f"tw__{bd[0]}_{bd[1]}__{ws}"] = _cov(seg[:, :, s:s+wlen])
    return views

# ---------- cache I/O ----------
def _write_manifest():
    COV_DIR.mkdir(parents=True, exist_ok=True)
    man = COV_DIR / "cache_manifest.json"
    if not man.exists():
        json.dump({
            "cov_signature": COV_SIG,
            "built_by": CONFIG["experiment_name"],
            "config_subset": COV_SUBSET,
            "array_shape": "(n_trials, n_channels, n_channels)  n_channels=29",
            "keys": {
                "fb__{lo}_{hi}": "full 0-4 s covariance for band [lo,hi]",
                "tw__{lo}_{hi}__{wstart}": "covariance over the 1-s window starting at {wstart}s for band [lo,hi]",
                "y": "labels (0=left,1=right)", "onsets": "per-trial MI onset sample", "_cov_sig": "signature string",
            },
            "bands": CONFIG["freq_bands"], "windows": WIN_STARTS, "win_len_s": CONFIG["time_window_len_s"],
        }, open(man,"w"), indent=2, default=str)

def save_subject_cov(sid, views, y, onsets):
    COV_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(COV_DIR/f"sub-{sid:02d}_cov.npz",
                        y=np.asarray(y), onsets=np.asarray(onsets), _cov_sig=np.array(COV_SIG), **views)
    _write_manifest()

def load_subject_cov(sid):
    p = COV_DIR/f"sub-{sid:02d}_cov.npz"
    if not p.exists(): return None
    z = np.load(p, allow_pickle=False)
    if "_cov_sig" in z.files and str(z["_cov_sig"]) != COV_SIG:
        print(f"  WARN sub-{sid:02d}: cache signature mismatch ({str(z['_cov_sig'])} != {COV_SIG})")
    views = {k: z[k] for k in z.files if k.startswith("fb__") or k.startswith("tw__")}
    return views, z["y"], z["onsets"]

def get_subject_covariances(sid, path):
    """Cache-aware. Returns (views, y, onsets, source)."""
    mode = CONFIG["covariance_cache_mode"]
    if mode in ("auto", "readonly"):
        cached = load_subject_cov(sid)
        if cached is not None:
            return cached[0], np.asarray(cached[1]).astype(int), np.asarray(cached[2]), "cache"
        if mode == "readonly":
            raise FileNotFoundError(f"readonly: no cache for sub-{sid:02d} under {COV_DIR}. Build it first (mode='rebuild'/'auto').")
    raw, y, onsets = load_subject_raw(path)
    views = build_subject_covariances(raw, onsets)
    if mode in ("auto", "rebuild"):
        save_subject_cov(sid, views, y, onsets)
    return views, y.astype(int), onsets, "computed"
print("Covariance builder + cache I/O defined.")

# 5. Protocols — leaky (max-on-test) vs honest (inner-CV on train)

In [ ]:
def _bacc(Ctr, ytr, Cte, yte):
    try:
        clf = FgMDM(metric=CONFIG["fgmdm_metric"]); clf.fit(Ctr, ytr)
        return float(balanced_accuracy_score(yte, clf.predict(Cte)))
    except Exception:
        return np.nan

def _acc_bacc(Ctr, ytr, Cte, yte):
    try:
        clf = FgMDM(metric=CONFIG["fgmdm_metric"]); clf.fit(Ctr, ytr); pred = clf.predict(Cte)
        return float(accuracy_score(yte, pred)), float(balanced_accuracy_score(yte, pred))
    except Exception:
        return np.nan, np.nan

def leaky_oracle(views, y, n_repeats, rng):
    """LEAKY (MATLAB): per view a fresh random 24/16 split, report max-over-views TEST accuracy (oracle)."""
    n = len(y); tr_n, te_n = CONFIG["train_size"], CONFIG["test_size"]; reps = []
    for _ in range(n_repeats):
        accs = []
        for cov in views.values():
            p = rng.permutation(n); tr = p[:tr_n]; te = p[tr_n:tr_n+te_n]
            a, _ = _acc_bacc(cov[tr], y[tr], cov[te], y[te]); accs.append(a)
        reps.append(np.nanmax(accs))
    return float(np.nanmean(reps))

def honest_nested(views, y, splits, inner_folds, rng_state):
    """HONEST: select the view by inner-CV balanced-acc on TRAIN only, eval once on the held-out test fold."""
    keys = list(views.keys()); accs, baccs, chosen = [], [], []
    for tr, te in splits:
        skf = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=rng_state)
        best_key, best_v = keys[0], -np.inf
        for k in keys:
            cov = views[k]
            iv = [_bacc(cov[tr[a]], y[tr[a]], cov[tr[b]], y[tr[b]]) for a, b in skf.split(tr, y[tr])]
            mv = np.nanmean(iv)
            if mv > best_v: best_v, best_key = mv, k
        a, ba = _acc_bacc(views[best_key][tr], y[tr], views[best_key][te], y[te])
        accs.append(a); baccs.append(ba); chosen.append(best_key)
    return float(np.nanmean(accs)), float(np.nanmean(baccs)), chosen
print("Protocols (leaky_oracle, honest_nested) defined.")

# 6. Run all subjects

In [ ]:
def make_splits(y):
    idx = np.arange(len(y))
    if CONFIG["cv_scheme"] == "kfold5":
        sp = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=CONFIG["random_state"])
    else:
        sp = StratifiedShuffleSplit(n_splits=CONFIG["n_repeats"], test_size=CONFIG["honest_test_frac"],
                                    random_state=CONFIG["random_state"])
    return list(sp.split(idx, y))

def run_all():
    files = find_subject_files(CONFIG["source_roots"])
    keep = CONFIG["subjects_to_use"]; rows = []; t0 = time.time()
    rng = np.random.RandomState(CONFIG["random_state"]); n_cache = n_comp = 0
    print("="*70)
    for path in files:
        sid = int(os.path.basename(path).split("-")[1][:2])
        if keep is not None and sid not in keep: continue
        views, y, onsets, source = get_subject_covariances(sid, path)
        n_cache += (source == "cache"); n_comp += (source == "computed")
        fb   = {k: v for k, v in views.items() if k.startswith("fb__")}
        twfb = {k: v for k, v in views.items() if k.startswith("tw__")}
        splits = make_splits(y)
        row = {"subject": sid, "n_trials": int(len(y)), "cov_source": source}
        if CONFIG["run_shipped_fb_leaky"]:
            row["shipped_fb_leaky"] = leaky_oracle(fb, y, CONFIG["n_repeats"], rng)
        if CONFIG["run_shipped_fb_honest"]:
            a, ba, _ = honest_nested(fb, y, splits, CONFIG["inner_folds"], 0)
            row["shipped_fb_honest_acc"], row["shipped_fb_honest_bacc"] = a, ba
        if CONFIG["run_twfb_leaky"]:
            row["twfb_leaky"] = leaky_oracle(twfb, y, CONFIG["n_repeats"], rng)
        if CONFIG["run_twfb_honest"]:
            a, ba, _ = honest_nested(twfb, y, splits, CONFIG["inner_folds"], 0)
            row["twfb_honest_acc"], row["twfb_honest_bacc"] = a, ba
        if CONFIG["run_perband_fixed"]:
            for k, cov in fb.items():
                row[f"fixed_{k.replace('fb__','')}"] = float(np.nanmean(
                    [_acc_bacc(cov[tr], y[tr], cov[te], y[te])[1] for tr, te in splits]))
        rows.append(row)
        shown = [c for c in ["shipped_fb_leaky","shipped_fb_honest_bacc","twfb_leaky","twfb_honest_bacc"] if c in row]
        print(f"  sub-{sid:02d} [{source:8s}]: " + "  ".join(f"{c.replace('_bacc','')}={row[c]*100:5.1f}" for c in shown)
              + f"   ({time.time()-t0:5.1f}s)")
    print("="*70); print(f"cov source: {n_cache} from cache, {n_comp} computed")
    return pd.DataFrame(rows)

RESULTS = run_all()
RESULTS.head()

# 7. Summary, artifacts, plots

In [ ]:
def _ms(df, col):
    v = df[col].dropna().values
    return [float(np.mean(v)), float(np.std(v, ddof=1) if len(v) > 1 else 0.0)]

PROTO_COLS = [("shipped_fb_leaky","Shipped .m (FB-only, leaky max-on-test)"),
              ("shipped_fb_honest_bacc","Shipped FB, honest nested-CV (bal.acc)"),
              ("twfb_leaky","Paper TWFB (TWxFB, leaky max-on-test)"),
              ("twfb_honest_bacc","TWFB honest nested-CV (bal.acc)")]

summary = {}
for col, label in PROTO_COLS:
    if col in RESULTS: summary[col] = _ms(RESULTS, col)

print("="*70); print(f"TWFB + DGFMDM (FgMDM) — Liu2024, n={len(RESULTS)} subjects")
for col, label in PROTO_COLS:
    if col in summary: print(f"  {label:42s}: {summary[col][0]*100:5.2f}%  ± {summary[col][1]*100:4.2f}")
print(f"  {'Liu2024 Table 4 TWFB+DGFMDRM (reported)':42s}: 72.21%")
if "twfb_leaky" in summary and "twfb_honest_bacc" in summary:
    gap = (summary["twfb_leaky"][0]-summary["twfb_honest_bacc"][0])*100
    print(f"  >>> LEAKAGE GAP (twfb leaky - honest): {gap:.1f} pts")
print("="*70)

RESULTS.to_csv(ARTIFACT_DIR/"subject_results.csv", index=False)
GLOBAL = {"experiment_name": CONFIG["experiment_name"], "n_subjects": int(len(RESULTS)),
          "cov_signature": COV_SIG, "cov_cache_dir": str(COV_DIR), **{k: summary[k] for k in summary}}
json.dump({"config": {k:(str(v) if isinstance(v,Path) else v) for k,v in CONFIG.items()},
           "summary": summary, "n_subjects": int(len(RESULTS))},
          open(ARTIFACT_DIR/"summary.json","w"), indent=2, default=str)
json.dump(GLOBAL, open(ARTIFACT_DIR/"global_metrics.json","w"), indent=2, default=str)
json.dump(RESULTS.to_dict(orient="records"), open(ARTIFACT_DIR/"subject_metrics.json","w"), indent=2, default=str)
json.dump({"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "cov_cache_dir": str(COV_DIR),
           "cov_signature": COV_SIG, "config": CONFIG, "n_subjects": int(len(RESULTS))},
          open(ARTIFACT_DIR/"run_metadata.json","w"), indent=2, default=str)

if HAVE_MPL:
    labels = [l for c,l in PROTO_COLS if c in summary]; cols = [c for c,l in PROTO_COLS if c in summary]
    vals = [summary[c][0]*100 for c in cols]; errs = [summary[c][1]*100 for c in cols]
    colors = ["#888","#4a4","#c44","#2a7"][:len(cols)]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(range(len(cols)), vals, yerr=errs, capsize=5, color=colors)
    ax.axhline(72.21, ls="--", c="purple", label="Liu Table 4 (72.21%)"); ax.axhline(50, ls=":", c="gray", label="chance")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c.replace('_bacc','') for c in cols], rotation=15, fontsize=8)
    ax.set_ylabel("accuracy / balanced accuracy (%)"); ax.set_title("Leaky vs honest"); ax.legend(fontsize=8)
    for i,v in enumerate(vals): ax.text(i, v+1, f"{v:.1f}", ha="center", fontsize=9)
    plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"leaky_vs_honest.png", dpi=200, bbox_inches="tight"); plt.close(fig)

    if "twfb_honest_bacc" in RESULTS:
        d = RESULTS.sort_values("subject")
        fig, ax = plt.subplots(figsize=(max(8,len(d)*0.4),4))
        ax.bar(d["subject"], d["twfb_honest_bacc"]*100, color="#2a7", alpha=0.85)
        ax.axhline(50, c="red", ls="--"); ax.axhline(d["twfb_honest_bacc"].mean()*100, c="orange",
                   label=f"mean={d['twfb_honest_bacc'].mean()*100:.1f}%")
        ax.set_xlabel("subject"); ax.set_ylabel("honest bal.acc (%)"); ax.legend()
        ax.set_xticks(d["subject"]); ax.set_xticklabels(d["subject"], rotation=90, fontsize=7)
        ax.set_title("Per-subject honest TWFB")
        plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"subject_performance.png", dpi=200, bbox_inches="tight"); plt.close(fig)
print("Saved subject_results.csv, summary.json, global_metrics.json, subject_metrics.json, run_metadata.json + PNGs")
print(f"Artifacts: {ARTIFACT_DIR}")